<a href="https://colab.research.google.com/github/ankur4020new-collab/Ankur-Yadav.2026B03110009/blob/main/Ankur_Yadav_2026B03110009.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Gym Membership & Attendance Analysis
**Project No.: 12**  
**Student Name:** Ankur Yadav  
**Admission No.:** 2026B03110009  
**Program Name:** Data Analysis  

## 1. Project overview
This project analyzes gym membership and attendance patterns to extract operational insights.

In [ ]:
# Import libraries and load the dataset
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Search for the CSV file dynamically
def find_file(name):
    for root, dirs, files in os.walk('.'):
        if name in files:
            return Path(root) / name
    for root, dirs, files in os.walk('/'):
        # Limit search depth or target specific folders to keep it fast
        if 'content' in root or 'data' in root:
            if name in files:
                return Path(root) / name
    return None

DATA_PATH = find_file("12_gym_membership.csv")
if DATA_PATH is None:
    # Fallback to defaults
    DATA_PATH = Path("../data/12_gym_membership.csv")
    if not DATA_PATH.exists():
        DATA_PATH = Path("12_gym_membership.csv")
    if not DATA_PATH.exists():
        DATA_PATH = Path("/content/12_gym_membership.csv")

print(f"Using dataset path: {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
display(df.head())

Using dataset path: 12_gym_membership.csv


,Checkin_ID,Member_ID,Checkin_Time,Workout_Duration_Mins,Membership_Tier,Fitness_Class
0,CH10000,M1103,2023-02-18 18:59:00,47.4,VIP,Yoga
1,CH10001,M1093,2023-06-22 10:01:00,52.9,Bronze,NaN
2,CH10002,M1015,2023-01-30 19:01:00,34.3,Gold,Spinning
3,CH10003,M1107,2023-05-24 12:25:00,78.1,Bronze,Spinning
4,CH10004,M1072,2023-06-17 17:12:00,NaN,Bronze,NaN


In [ ]:
import os
from pathlib import Path

def search_csv(filename):
    matches = []
    for root, dirs, files in os.walk('.'):
        if filename in files:
            matches.append(os.path.join(root, filename))
    for root, dirs, files in os.walk('/content'):
        if filename in files:
            matches.append(os.path.join(root, filename))
    return list(set(matches))

found = search_csv('12_gym_membership.csv')
print("Found paths for dataset:", found)

Found paths for dataset: ['/content/12_gym_membership.csv', './12_gym_membership.csv']


In [ ]:
# Data quality inspection
print("Column names:", df.columns.tolist())
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing_Count"))
print("Exact duplicate rows:", df.duplicated().sum())

Column names: ['Checkin_ID', 'Member_ID', 'Checkin_Time', 'Workout_Duration_Mins', 'Membership_Tier', 'Fitness_Class']

Missing values:


,Missing_Count
Checkin_ID,0
Member_ID,0
Checkin_Time,0
Workout_Duration_Mins,50
Membership_Tier,0
Fitness_Class,307


Exact duplicate rows: 0


In [ ]:
# Clean and transform
gym = df.copy()
gym["Checkin_Time"] = pd.to_datetime(gym["Checkin_Time"], errors="coerce")
gym = gym.drop_duplicates(subset=["Checkin_ID"]).copy()

duration_median = gym["Workout_Duration_Mins"].median()
gym["Workout_Duration_Mins"] = gym["Workout_Duration_Mins"].fillna(duration_median)

gym["Checkin_Hour"] = gym["Checkin_Time"].dt.hour
gym["Checkin_Weekday"] = gym["Checkin_Time"].dt.day_name()
gym["Checkin_Month"] = gym["Checkin_Time"].dt.strftime("%B")

display(gym.head())

,Checkin_ID,Member_ID,Checkin_Time,Workout_Duration_Mins,Membership_Tier,Fitness_Class,Checkin_Hour,Checkin_Weekday,Checkin_Month
0,CH10000,M1103,2023-02-18 18:59:00,47.4,VIP,Yoga,18,Saturday,February
1,CH10001,M1093,2023-06-22 10:01:00,52.9,Bronze,NaN,10,Thursday,June
2,CH10002,M1015,2023-01-30 19:01:00,34.3,Gold,Spinning,19,Monday,January
3,CH10003,M1107,2023-05-24 12:25:00,78.1,Bronze,Spinning,12,Wednesday,May
4,CH10004,M1072,2023-06-17 17:12:00,55.1,Bronze,NaN,17,Saturday,June


In [ ]:
# Key performance indicators
kpis = pd.Series({
    "Original check-in records": len(df),
    "Cleaned check-in records": len(gym),
    "Unique members": gym["Member_ID"].nunique(),
    "Membership tiers": gym["Membership_Tier"].nunique(),
    "Average workout duration (mins)": gym["Workout_Duration_Mins"].mean()
})
display(kpis.to_frame("Value"))

,Value
Original check-in records,1000.0000
Cleaned check-in records,1000.0000
Unique members,150.0000
Membership tiers,4.0000
Average workout duration (mins),55.1562


In [ ]:
# Membership tier and fitness-class summaries
tier_summary = (gym.groupby("Membership_Tier")
                .agg(Checkins=("Checkin_ID", "count"),
                     Unique_Members=("Member_ID", "nunique"),
                     Avg_Duration=("Workout_Duration_Mins", "mean")))
display(tier_summary)

class_summary = (gym.groupby("Fitness_Class")
                 .agg(Checkins=("Checkin_ID", "count"),
                      Avg_Duration=("Workout_Duration_Mins", "mean")))
display(class_summary)

,Checkins,Unique_Members,Avg_Duration
Membership_Tier,,,
Bronze,422,142,55.795024
Gold,188,106,55.418617
Silver,292,126,54.308219
VIP,98,76,54.428571


,Checkins,Avg_Duration
Fitness_Class,,
HIIT,210,55.805238
Spinning,196,55.445408
Yoga,149,52.267114
Zumba,138,55.036232


In [ ]:
# Attendance patterns by hour, weekday and month
hour_summary = gym.groupby("Checkin_Hour").agg(Checkins=("Checkin_ID", "count"))
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_summary = (gym.groupby("Checkin_Weekday")
                   .agg(Checkins=("Checkin_ID", "count"))
                   .reindex(weekday_order))

month_summary = gym.groupby("Checkin_Month").agg(Checkins=("Checkin_ID", "count"))

display(hour_summary)
display(weekday_summary)
display(month_summary)

,Checkins
Checkin_Hour,
6,18
7,55
8,91
9,96
10,101
11,45
12,48
13,31
14,33


,Checkins
Checkin_Weekday,
Monday,131
Tuesday,138
Wednesday,154
Thursday,163
Friday,128
Saturday,145
Sunday,141


,Checkins
Checkin_Month,
April,161
February,168
January,186
June,159
March,162
May,164


In [ ]:
# Member-level check-in activity (descriptive, not membership duration)
member_activity = (gym.groupby("Member_ID")
                   .agg(Checkins=("Checkin_ID", "count"),
                        Avg_Duration=("Workout_Duration_Mins", "mean")))
display(member_activity.describe())

,Checkins,Avg_Duration
count,150.000000,150.000000
mean,6.666667,54.998586
std,2.721199,6.670178
min,1.000000,35.866667
25%,5.000000,50.890455
50%,6.000000,55.203571
75%,8.000000,58.836250
max,15.000000,80.400000


In [ ]:
# NumPy descriptive statistics and IQR outlier detection
durations = gym["Workout_Duration_Mins"].dropna().to_numpy()
duration_stats = {
    "Mean": np.mean(durations),
    "Median": np.median(durations),
    "Std Dev": np.std(durations),
    "Min": np.min(durations),
    "Max": np.max(durations)
}
print("Descriptive statistics of duration (NumPy):", duration_stats)

q75, q25 = np.percentile(durations, [75, 25])
iqr = q75 - q25
lower_bound = q25 - 1.5 * iqr
upper_bound = q75 + 1.5 * iqr
outliers = durations[(durations < lower_bound) | (durations > upper_bound)]
print(f"IQR: {iqr}, Outliers detected count: {len(outliers)}")

Descriptive statistics of duration (NumPy): {'Mean': np.float64(55.156200000000005), 'Median': np.float64(55.1), 'Std Dev': np.float64(14.566015294513459), 'Min': np.float64(15.0), 'Max': np.float64(95.3)}
IQR: 20.1, Outliers detected count: 3


In [ ]:
# Relationship between membership tier and workout duration
tier_duration = gym.groupby("Membership_Tier")["Workout_Duration_Mins"].agg(["count", "mean", "median"])
display(tier_duration)
print("Correlation of numeric attributes:")
display(gym[["Workout_Duration_Mins", "Checkin_Hour"]].corr())

,count,mean,median
Membership_Tier,,,
Bronze,422,55.795024,55.1
Gold,188,55.418617,55.1
Silver,292,54.308219,55.1
VIP,98,54.428571,55.1


Correlation of numeric attributes:


,Workout_Duration_Mins,Checkin_Hour
Workout_Duration_Mins,1.000000,0.037083
Checkin_Hour,0.037083,1.000000


In [ ]:
# Show attendance and class distribution calculations instead of graphs
print("--- Attendance Peak Hours (Sorted by Checkins) ---")
display(hour_summary.sort_values(by='Checkins', ascending=False))

print("\n--- Fitness Class Distribution Calculation ---")
display(class_summary)

--- Attendance Peak Hours (Sorted by Checkins) ---


,Checkins
Checkin_Hour,
18,124
19,104
10,101
9,96
8,91
17,86
7,55
16,49
12,48



--- Fitness Class Distribution Calculation ---


,Checkins,Avg_Duration
Fitness_Class,,
HIIT,210,55.805238
Spinning,196,55.445408
Yoga,149,52.267114
Zumba,138,55.036232


In [ ]:
# Export cleaned data and analysis tables
OUTPUT_DIR = Path("../outputs")
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gym.to_csv(OUTPUT_DIR / "cleaned_gym_membership.csv", index=False)
kpis.to_frame("Value").to_csv(OUTPUT_DIR / "kpis.csv")
tier_summary.to_csv(OUTPUT_DIR / "tier_summary.csv")
class_summary.to_csv(OUTPUT_DIR / "class_summary.csv")
print("Exported datasets and tables to:", OUTPUT_DIR.resolve())

Exported datasets and tables to: /content/outputs


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def get_dataset():
    # Search for files ending with or matching gym_membership.csv
    for root, dirs, files in os.walk('.'):
        for file in files:
            if 'gym_membership' in file and file.endswith('.csv'):
                return Path(root) / file
    for root, dirs, files in os.walk('/content'):
        for file in files:
            if 'gym_membership' in file and file.endswith('.csv'):
                return Path(root) / file
    return None

path = get_dataset()
if path:
    print(f'Located dataset at: {path}')
    df = pd.read_csv(path)

    # Clean dataset
    gym = df.copy()
    gym['Checkin_Time'] = pd.to_datetime(gym['Checkin_Time'], errors='coerce')
    gym = gym.drop_duplicates(subset=['Checkin_ID']).copy()
    duration_median = gym['Workout_Duration_Mins'].median()
    gym['Workout_Duration_Mins'] = gym['Workout_Duration_Mins'].fillna(duration_median)
    gym['Checkin_Hour'] = gym['Checkin_Time'].dt.hour
    gym['Checkin_Weekday'] = gym['Checkin_Time'].dt.day_name()
    gym['Checkin_Month'] = gym['Checkin_Time'].dt.strftime('%B')

    # KPIs
    kpis = pd.Series({
        'Original check-in records': len(df),
        'Cleaned check-in records': len(gym),
        'Unique members': gym['Member_ID'].nunique(),
        'Membership tiers': gym['Membership_Tier'].nunique(),
        'Average workout duration (mins)': gym['Workout_Duration_Mins'].mean()
    })
    display(kpis.to_frame('Value'))

    # Summaries
    tier_summary = (gym.groupby('Membership_Tier')
                    .agg(Checkins=('Checkin_ID', 'count'),
                         Unique_Members=('Member_ID', 'nunique'),
                         Avg_Duration=('Workout_Duration_Mins', 'mean')))
    display(tier_summary)

    class_summary = (gym.groupby('Fitness_Class')
                     .agg(Checkins=('Checkin_ID', 'count'),
                          Avg_Duration=('Workout_Duration_Mins', 'mean')))
    display(class_summary)

    # Export
    OUTPUT_DIR = Path('outputs')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    gym.to_csv(OUTPUT_DIR / 'cleaned_gym_membership.csv', index=False)
    kpis.to_frame('Value').to_csv(OUTPUT_DIR / 'kpis.csv')
    tier_summary.to_csv(OUTPUT_DIR / 'tier_summary.csv')
    class_summary.to_csv(OUTPUT_DIR / 'class_summary.csv')
    print('All tables successfully saved to outputs directory.')
else:
    print('The dataset 12_gym_membership.csv could not be located in this environment. Please ensure the file is uploaded.')

Located dataset at: 12_gym_membership.csv


,Value
Original check-in records,1000.0000
Cleaned check-in records,1000.0000
Unique members,150.0000
Membership tiers,4.0000
Average workout duration (mins),55.1562


,Checkins,Unique_Members,Avg_Duration
Membership_Tier,,,
Bronze,422,142,55.795024
Gold,188,106,55.418617
Silver,292,126,54.308219
VIP,98,76,54.428571


,Checkins,Avg_Duration
Fitness_Class,,
HIIT,210,55.805238
Spinning,196,55.445408
Yoga,149,52.267114
Zumba,138,55.036232


All tables successfully saved to outputs directory.
